# Trend Category SPAN Analysis

Classify the top-50 corpus keywords per trend category (Emerging / Stable / Decaying)
directly from the **full corpus TF-IDF**, then compute their SPAN across all 5 models.

**Output:**
- `trend_category_keywords.csv` — the 150 classified keywords (50 per category)
- `trend_category_span.csv` — SPAN per keyword × model with category label
- `trend_category_span_summary.csv` — avg-SPAN per model × category (paper + simple)

In [8]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")

## Configuration

In [9]:
BASE = Path("/home/nedo/Kuliah/TA/Program")
DATA_DIR = BASE / "data" / "preprocess"
RESULTS_DIR = BASE / "results"

LIST_SUBJECT = ["cs", "math", "physics"]
MODELS = ["dtm", "lda", "top2vec", "bertopic", "topicGpt"]
MODEL_LABELS = {
    "dtm": "DTM", "lda": "LDA", "top2vec": "Top2Vec",
    "bertopic": "BERTopic", "topicGpt": "TopicGPT",
}

TOP_K = 50  # top keywords per trend category

for subject in LIST_SUBJECT:
    (RESULTS_DIR / "shared" / "tren" / subject).mkdir(parents=True, exist_ok=True)

print(f"Models: {list(MODEL_LABELS.values())}")
print(f"Top-K per trend category: {TOP_K}")
print(f"Subjects: {LIST_SUBJECT}")

Models: ['DTM', 'LDA', 'Top2Vec', 'BERTopic', 'TopicGPT']
Top-K per trend category: 50
Subjects: ['cs', 'math', 'physics']


## Helper Functions

In [10]:
import re
TOKEN_PATTERN = re.compile(r"(?u)\b\w\w+\b")

def compute_corpus_word_freq(subject):
    """
    Count each word across all documents in corpus (v̂ₖ).
    Uses emb/v1.csv with sklearn's default tokenizer pattern
    to match TfidfVectorizer vocabulary.
    """
    df = pd.read_csv(DATA_DIR / subject / "emb/v1.csv")
    word_freq = defaultdict(int)
    for text_val in df["text"]:
        try:
            tokens = ast.literal_eval(text_val)
            if isinstance(tokens, list):
                text_str = " ".join(tokens)
            else:
                text_str = str(text_val)
        except (ValueError, SyntaxError):
            text_str = str(text_val)
        for w in TOKEN_PATTERN.findall(text_str.lower()):
            word_freq[w] += 1
    return word_freq


def load_topic_words_by_year(model, subject):
    """Load topic-word evolution → {year: [set of words, ...]}."""
    evo_path = RESULTS_DIR / model / "temporal" / subject / "topic_word_evolution.csv"
    if not evo_path.exists():
        return {}, []
    evo_df = pd.read_csv(evo_path)
    years = sorted(evo_df["year"].unique())
    topic_words_by_year = defaultdict(list)
    for _, row in evo_df.iterrows():
        words = set(w.strip() for w in str(row["top_words"]).split(","))
        topic_words_by_year[int(row["year"])].append(words)
    return topic_words_by_year, years


def compute_span(keyword, topic_words_by_year, years):
    """SPAN = longest consecutive years keyword appears in any topic."""
    trend = []
    for y in years:
        found = any(keyword in words for words in topic_words_by_year.get(y, []))
        trend.append(1 if found else 0)
    max_span = 0
    current = 0
    for t in trend:
        if t == 1:
            current += 1
            max_span = max(max_span, current)
        else:
            current = 0
    return max_span, trend


def compute_yearly_tfidf(subject):
    """Compute average TF-IDF score per word per year from raw documents."""
    df = pd.read_csv(DATA_DIR / subject / "emb/v1.csv")
    df["year"] = pd.to_datetime(df["submitted_date"]).dt.year

    def to_text(val):
        try:
            tokens = ast.literal_eval(val)
            if isinstance(tokens, list):
                return " ".join(tokens)
        except (ValueError, SyntaxError):
            pass
        return str(val)

    df["text_str"] = df["text"].apply(to_text)
    years = sorted(df["year"].unique())

    yearly_scores = {}
    for year in years:
        year_docs = df[df["year"] == year]["text_str"].tolist()
        if len(year_docs) < 5:
            continue
        tfidf = TfidfVectorizer(max_features=5000, min_df=2, stop_words="english")
        matrix = tfidf.fit_transform(year_docs)
        feature_names = tfidf.get_feature_names_out()
        avg_scores = np.asarray(matrix.mean(axis=0)).flatten()
        for word, score in zip(feature_names, avg_scores):
            if word not in yearly_scores:
                yearly_scores[word] = {}
            yearly_scores[word][year] = float(score)

    return yearly_scores, years


def classify_keywords(yearly_scores, years, top_k=50):
    """
    Classify ALL corpus words using linear regression slope of TF-IDF over time.
    Returns top_k per category: emerging, stable, decaying.
    """
    word_stats = []
    years_arr = np.array(years, dtype=float)

    for word, scores in yearly_scores.items():
        vals = np.array([scores.get(y, 0.0) for y in years])

        n_present = np.sum(vals > 0)
        if n_present < 3:
            continue

        slope, intercept, r_val, p_val, std_err = linregress(years_arr, vals)
        overall_avg = vals.mean()

        word_stats.append({
            "word": word,
            "slope": slope,
            "overall_avg": overall_avg,
            "r_squared": r_val ** 2,
        })

    stats_df = pd.DataFrame(word_stats)

    # Emerging: highest positive slope
    emerging_pool = stats_df[stats_df["slope"] > 0]
    emerging = emerging_pool.nlargest(top_k, "slope")

    # Stable: high avg TF-IDF + lowest absolute slope
    used = set(emerging["word"])
    stable_pool = stats_df[~stats_df["word"].isin(used)].copy()
    stable_pool["stability"] = stable_pool["overall_avg"] / (
        1 + stable_pool["slope"].abs() * 10000
    )
    stable = stable_pool.nlargest(top_k, "stability")

    # Decaying: strongest negative slope
    used.update(stable["word"])
    decay_pool = stats_df[
        (~stats_df["word"].isin(used)) & (stats_df["slope"] < 0)
    ]
    decaying = decay_pool.nsmallest(top_k, "slope")

    return emerging, stable, decaying

## Step 1: Load Model Data

In [11]:
all_model_data = {}   # {subject: {model: (tw_by_year, years)}}
all_corpus_freq = {}  # {subject: {word: count}}

for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Loading: {subject.upper()}")
    print(f"{'='*70}")

    model_data = {}
    for model in MODELS:
        tw, years = load_topic_words_by_year(model, subject)
        model_data[model] = (tw, years)
        n_terms = sum(len(ws) for year_list in tw.values() for ws in year_list)
        print(f"  {MODEL_LABELS[model]:>10s}: {len(years)} years, {n_terms:,} term-occurrences")

    # Compute corpus frequency
    print(f"  Computing corpus word frequencies...")
    corpus_freq = compute_corpus_word_freq(subject)
    print(f"  Corpus vocabulary: {len(corpus_freq):,} unique words")

    all_model_data[subject] = model_data
    all_corpus_freq[subject] = corpus_freq


Loading: CS
         DTM: 26 years, 13,000 term-occurrences
         LDA: 26 years, 16,760 term-occurrences
     Top2Vec: 26 years, 50,500 term-occurrences
    BERTopic: 26 years, 43,280 term-occurrences
    TopicGPT: 26 years, 22,210 term-occurrences
  Computing corpus word frequencies...
  Corpus vocabulary: 166,494 unique words

Loading: MATH
         DTM: 26 years, 13,000 term-occurrences
         LDA: 26 years, 12,860 term-occurrences
     Top2Vec: 26 years, 52,100 term-occurrences
    BERTopic: 26 years, 35,720 term-occurrences
    TopicGPT: 26 years, 15,510 term-occurrences
  Computing corpus word frequencies...
  Corpus vocabulary: 110,120 unique words

Loading: PHYSICS
         DTM: 26 years, 15,600 term-occurrences
         LDA: 26 years, 12,930 term-occurrences
     Top2Vec: 26 years, 49,810 term-occurrences
    BERTopic: 26 years, 51,620 term-occurrences
    TopicGPT: 26 years, 17,530 term-occurrences
  Computing corpus word frequencies...
  Corpus vocabulary: 140,342 uniq

## Step 2: Classify Top-50 Keywords per Category (from Full Corpus)

Use TF-IDF slope over time to classify the **full corpus vocabulary** into:
- **Emerging** (50): strongest positive slope
- **Stable** (50): high avg TF-IDF with near-zero slope
- **Decaying** (50): strongest negative slope

In [12]:
keyword_cache = {}  # {subject: (emerging_df, stable_df, decaying_df)}

for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"TF-IDF Classification: {subject.upper()}")
    print(f"{'='*70}")

    print(f"  Computing yearly TF-IDF...")
    yearly_scores, years = compute_yearly_tfidf(subject)
    print(f"  {len(yearly_scores):,} unique words tracked")

    # Classify from FULL corpus vocabulary (not shared keywords)
    emerging, stable, decaying = classify_keywords(
        yearly_scores, years, top_k=TOP_K
    )
    keyword_cache[subject] = (emerging, stable, decaying)

    # Save trend_category_keywords.csv
    out_dir = RESULTS_DIR / "shared" / "tren" / subject
    kw_rows = []
    for cat, cat_df in [("emerging", emerging), ("stable", stable), ("decaying", decaying)]:
        for _, r in cat_df.iterrows():
            kw_rows.append({
                "word": r["word"],
                "category": cat,
                "slope": round(r["slope"], 8),
                "overall_avg": round(r["overall_avg"], 8),
            })
    kw_df = pd.DataFrame(kw_rows)
    kw_df.to_csv(out_dir / "trend_category_keywords.csv", index=False)

    for cat, icon, cat_df in [
        ("Emerging", "📈", emerging),
        ("Stable", "🔒", stable),
        ("Decaying", "📉", decaying),
    ]:
        print(f"\n  {icon} {cat} ({len(cat_df)} keywords):")
        print(f"  {'Word':25s} {'Slope':>12s} {'Avg TF-IDF':>12s}")
        print(f"  {'-'*50}")
        for _, r in cat_df.head(10).iterrows():
            print(f"  {r['word']:25s} {r['slope']:12.6f} {r['overall_avg']:12.6f}")
        if len(cat_df) > 10:
            print(f"  ... ({len(cat_df) - 10} more)")

    print(f"\n  Saved: {out_dir / 'trend_category_keywords.csv'}")


TF-IDF Classification: CS
  Computing yearly TF-IDF...
  12,598 unique words tracked

  📈 Emerging (50 keywords):
  Word                             Slope   Avg TF-IDF
  --------------------------------------------------
  learning                      0.000928     0.015563
  deep                          0.000662     0.005554
  training                      0.000656     0.006921
  models                        0.000630     0.013371
  image                         0.000608     0.008584
  neural                        0.000560     0.007821
  dataset                       0.000544     0.004424
  datasets                      0.000485     0.004442
  detection                     0.000456     0.007047
  tasks                         0.000451     0.005846
  ... (40 more)

  🔒 Stable (50 keywords):
  Word                             Slope   Avg TF-IDF
  --------------------------------------------------
  results                       0.000015     0.012940
  using                         0.

## Step 3: Compute SPAN per Category Keyword × Model

In [13]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Trend Category SPAN: {subject.upper()}")
    print(f"{'='*70}")

    model_data = all_model_data[subject]
    corpus_freq = all_corpus_freq[subject]
    emerging, stable, decaying = keyword_cache[subject]
    ref_years = model_data["dtm"][1]

    # Combine all classified keywords
    all_keywords = []
    for cat, cat_df in [("emerging", emerging), ("stable", stable), ("decaying", decaying)]:
        for _, r in cat_df.iterrows():
            all_keywords.append((r["word"], cat))

    print(f"  Total classified keywords: {len(all_keywords)}")

    # Compute SPAN for each keyword × model
    rows = []
    for word, category in all_keywords:
        v_hat = corpus_freq.get(word, 0)
        row = {"word": word, "category": category, "v_hat": v_hat}

        for model in MODELS:
            tw, years = model_data[model]
            span, trend = compute_span(word, tw, years)
            s_dict = span / v_hat if v_hat > 0 else 0.0
            label = MODEL_LABELS[model]
            row[f"span_{label}"] = span
            row[f"trend_{label}"] = str(trend)
            row[f"s_dict_{label}"] = round(s_dict, 6)

        rows.append(row)

    cat_span_df = pd.DataFrame(rows)

    out_dir = RESULTS_DIR / "shared" / "tren" / subject
    cat_span_df.to_csv(out_dir / "trend_category_span.csv", index=False)

    # Print cross-model comparison per category
    span_cols = [f"span_{MODEL_LABELS[m]}" for m in MODELS]
    for cat, icon in [("emerging", "📈"), ("stable", "🔒"), ("decaying", "📉")]:
        cat_data = cat_span_df[cat_span_df["category"] == cat]
        print(f"\n  {icon} {cat.upper()} (top 10 of {len(cat_data)}):")
        hdr = f"  {'Word':<20s} {'v̂ₖ':>8s}  " + "  ".join(f"{MODEL_LABELS[m]:>8s}" for m in MODELS)
        print(hdr)
        for _, r in cat_data.head(10).iterrows():
            spans_str = "  ".join(f"{r[c]:>8d}" for c in span_cols)
            print(f"  {r['word']:<20s} {r['v_hat']:>8d}  {spans_str}")

    print(f"\n  Saved: {out_dir / 'trend_category_span.csv'}")


Trend Category SPAN: CS
  Total classified keywords: 150

  📈 EMERGING (top 10 of 50):
  Word                      v̂ₖ       DTM       LDA   Top2Vec  BERTopic  TopicGPT
  learning               121400        12        26        26        13        26
  deep                    37999        11        15        16        12        15
  training                55660        12        17        14         3        13
  models                 107355         0        26        23         6        14
  image                   47072        17        24        24        18        20
  neural                  45675         9        24        24        15        16
  dataset                 34028        13        15         7         3         7
  datasets                35156         0        14         3         1         4
  detection               34610        12        18        18        17        15
  tasks                   41179         0        14        17         9        15

  🔒 STABL

## Step 4: avg-SPAN Summary per Model × Category

| Metric | Formula | Description |
|--------|---------|-------------|
| **avg-SPAN (paper)** | (1/N_captured) × Σ (Sₖ/v̂ₖ) | Rewards rare-but-persistent terms |
| **avg-SPAN (simple)** | (1/N_captured) × Σ Sₖ | Simple mean of raw SPAN |

Both averages computed over captured keywords only (SPAN > 0).

In [14]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Trend Category avg-SPAN: {subject.upper()}")
    print(f"{'='*70}")

    out_dir = RESULTS_DIR / "shared" / "tren" / subject
    cat_span_df = pd.read_csv(out_dir / "trend_category_span.csv")

    summary_rows = []

    for model in MODELS:
        label = MODEL_LABELS[model]
        spans_col = f"span_{label}"
        s_dict_col = f"s_dict_{label}"

        for cat in ["all", "emerging", "stable", "decaying"]:
            if cat == "all":
                subset = cat_span_df
            else:
                subset = cat_span_df[cat_span_df["category"] == cat]

            n_keywords = len(subset)
            spans = subset[spans_col]
            s_dicts = subset[s_dict_col]

            n_captured = int((spans > 0).sum())
            captured_mask = spans > 0

            avg_span_paper = s_dicts[captured_mask].mean() if n_captured > 0 else 0.0
            avg_span_simple = spans[captured_mask].mean() if n_captured > 0 else 0.0

            summary_rows.append({
                "model": label,
                "category": cat,
                "n_keywords": n_keywords,
                "n_captured": n_captured,
                "capture_pct": round(n_captured / n_keywords * 100, 2) if n_keywords > 0 else 0.0,
                "avg_span_paper": round(avg_span_paper, 8),
                "avg_span_simple": round(avg_span_simple, 4),
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(out_dir / "trend_category_span_summary.csv", index=False)

    # Print: overall first
    print(f"\n  Overall (all categories):")
    print(f"  {'Model':<10s} {'avg-SPAN(paper)':>16s} {'avg-SPAN(simple)':>16s} {'Captured':>12s}")
    print(f"  {'-'*58}")
    overall = summary_df[summary_df["category"] == "all"]
    for _, s in overall.iterrows():
        print(f"  {s['model']:<10s} {s['avg_span_paper']:>16.8f} {s['avg_span_simple']:>16.4f} "
              f"{s['n_captured']:>5d} ({s['capture_pct']:>5.1f}%)")

    # Per-category breakdown
    for cat, icon in [("emerging", "📈"), ("stable", "🔒"), ("decaying", "📉")]:
        cat_summary = summary_df[summary_df["category"] == cat]
        n_kw = cat_summary["n_keywords"].iloc[0]
        print(f"\n  {icon} {cat.upper()} ({n_kw} keywords):")
        print(f"  {'Model':<10s} {'avg-SPAN(paper)':>16s} {'avg-SPAN(simple)':>16s} {'Captured':>12s}")
        print(f"  {'-'*58}")
        for _, s in cat_summary.iterrows():
            print(f"  {s['model']:<10s} {s['avg_span_paper']:>16.8f} {s['avg_span_simple']:>16.4f} "
                  f"{s['n_captured']:>5d} ({s['capture_pct']:>5.1f}%)")

    print(f"\n  Saved: {out_dir / 'trend_category_span_summary.csv'}")

print("\n✅ All done!")


Trend Category avg-SPAN: CS

  Overall (all categories):
  Model       avg-SPAN(paper) avg-SPAN(simple)     Captured
  ----------------------------------------------------------
  DTM              0.00075547          12.7922    77 ( 51.3%)
  LDA              0.00128428          16.1716   134 ( 89.3%)
  Top2Vec          0.00153805          16.8182   132 ( 88.0%)
  BERTopic         0.00103589          10.6212   132 ( 88.0%)
  TopicGPT         0.00092981          11.8033   122 ( 81.3%)

  📈 EMERGING (50 keywords):
  Model       avg-SPAN(paper) avg-SPAN(simple)     Captured
  ----------------------------------------------------------
  DTM              0.00039556          10.4800    25 ( 50.0%)
  LDA              0.00053527          15.9333    45 ( 90.0%)
  Top2Vec          0.00053993          15.7500    44 ( 88.0%)
  BERTopic         0.00037310           9.4762    42 ( 84.0%)
  TopicGPT         0.00041556          12.1220    41 ( 82.0%)

  🔒 STABLE (50 keywords):
  Model       avg-SPAN(p